# Healthcare Claims - Bronze to Silver Transformations

This notebook reads raw data from the Bronze layer and applies cleansing, deduplication, and standardization to produce Silver tables.

**Dependency**: Runs the setup notebook first via `%run`.

In [0]:
%run ./healthcare_claims_setup

## Silver Layer - Data Cleansing and Standardization

Transformations applied:
- Remove duplicates
- Trim whitespace from string columns
- Handle null values
- Validate date ranges
- Add metadata columns (processed_timestamp, data_quality_flag)

In [0]:
# Bronze to Silver: medical_claim table
# Transformations: deduplication, null handling, date validation

medical_claim_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.medical_claim")

# Apply transformations
medical_claim_silver = (
    medical_claim_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("location_of_care", F.trim(F.col("location_of_care")))
    .withColumn("pay_type", F.trim(F.col("pay_type")))
    # Data quality flag: check for nulls in critical fields and date validity
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("date_service").isNull()) |
            (F.col("date_service_end") < F.col("date_service")),
            "FAIL"
        ).otherwise("PASS")
    )
    # Add processing timestamp
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates based on claim_id and patient_id
    .dropDuplicates(["claim_id", "patient_id"])
)

# Write to Silver layer
medical_claim_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")
print(f"  Records processed: {medical_claim_silver.count():,}")

In [0]:
# Bronze to Silver: diagnosis table
# Transformations: deduplication, standardization, null handling

diagnosis_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.diagnosis")

diagnosis_silver = (
    diagnosis_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("diagnosis_code", F.trim(F.upper(F.col("diagnosis_code"))))  # Standardize to uppercase
    .withColumn("diagnosis_qual", F.trim(F.col("diagnosis_qual")))
    .withColumn("admit_diagnosis_ind", F.trim(F.upper(F.col("admit_diagnosis_ind"))))
    # Data quality checks
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("diagnosis_code").isNull()) |
            (F.col("date_service_end") < F.col("date_service")),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "diagnosis_code"])
)

diagnosis_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis")
print(f"  Records processed: {diagnosis_silver.count():,}")

In [0]:
# Bronze to Silver: procedure table
# Transformations: deduplication, numeric validation, standardization

procedure_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.procedure")

procedure_silver = (
    procedure_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("service_line_number", F.trim(F.col("service_line_number")))
    .withColumn("procedure_code", F.trim(F.upper(F.col("procedure_code"))))
    .withColumn("procedure_qual", F.trim(F.col("procedure_qual")))
    .withColumn("revenue_code", F.trim(F.col("revenue_code")))
    # Standardize modifiers
    .withColumn("procedure_modifier1", F.trim(F.upper(F.col("procedure_modifier1"))))
    .withColumn("procedure_modifier2", F.trim(F.upper(F.col("procedure_modifier2"))))
    .withColumn("procedure_modifier3", F.trim(F.upper(F.col("procedure_modifier3"))))
    .withColumn("procedure_modifier4", F.trim(F.upper(F.col("procedure_modifier4"))))
    # Data quality checks: validate numeric fields and critical columns
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("procedure_code").isNull()) |
            (F.col("date_service_end") < F.col("date_service")) |
            (F.col("line_charge") < 0) |
            (F.col("line_allowed") < 0),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "service_line_number"])
)

procedure_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
print(f"  Records processed: {procedure_silver.count():,}")

In [0]:
# Bronze to Silver: provider table
# Transformations: deduplication, NPI standardization

provider_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.provider")

provider_silver = (
    provider_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("npi", F.trim(F.col("npi")))
    .withColumn("npi_role", F.trim(F.upper(F.col("npi_role"))))
    .withColumn("taxonomy_code", F.trim(F.col("taxonomy_code")))
    # Data quality checks: NPI should be 10 digits
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("npi").isNull()) |
            (F.length(F.col("npi")) != 10),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "npi", "npi_role"])
)

provider_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.provider"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.provider")
print(f"  Records processed: {provider_silver.count():,}")

In [0]:
# Bronze to Silver: enrollment table
# Transformations: deduplication, demographic standardization, date validation

enrollment_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.enrollment")

enrollment_silver = (
    enrollment_bronze
    # Trim string columns
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("patient_gender", F.trim(F.upper(F.col("patient_gender"))))
    .withColumn("patient_year_of_birth", F.trim(F.col("patient_year_of_birth")))
    .withColumn("patient_zip3", F.trim(F.col("patient_zip3")))
    .withColumn("patient_state", F.trim(F.upper(F.col("patient_state"))))
    .withColumn("benefit_type", F.trim(F.upper(F.col("benefit_type"))))
    .withColumn("pay_type", F.trim(F.upper(F.col("pay_type"))))
    # Calculate age (approximate based on current year)
    .withColumn(
        "patient_age",
        F.when(
            F.col("patient_year_of_birth").isNotNull(),
            F.year(F.current_date()) - F.col("patient_year_of_birth").cast("int")
        )
    )
    # Data quality checks
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("patient_id").isNull()) | 
            (F.col("date_start").isNull()) |
            (F.col("date_end") < F.col("date_start")) |
            (F.col("patient_age") < 0) |
            (F.col("patient_age") > 120),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["patient_id", "date_start"])
)

enrollment_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment")
print(f"  Records processed: {enrollment_silver.count():,}")